# Synthea Claims Analytics Platform

A walkthrough of the full analytics stack: ingest → star schema → analytics → risk scoring → care-gap explanation.

**Data source:** [Synthea](https://synthea.mitre.org) — an open-source synthetic patient generator
from MITRE Corporation. Synthea produces realistic but entirely fake patient records.
No real patient data is used here, but we treat it as PHI to demonstrate regulated-data discipline.

> **Why synthetic data?** Real Medicare claims require a CMS Data Use Agreement and take months
> to obtain. Synthea lets us demonstrate the full pipeline architecture without any data-access barriers.
> The real-data migration path (Blue Button 2.0 FHIR API) is documented in `ARCHITECTURE.md`.

In [ ]:
import sys
import pathlib

# Add project src to path so we can import cms_platform
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import duckdb
import polars as pl

print(f'duckdb  {duckdb.__version__}')
print(f'polars  {pl.__version__}')

## Section 1 — Ingest

Synthea exports five CSV files. We load them into raw DuckDB tables with **all columns as VARCHAR** —
no type inference at ingest time. Type casting happens in the schema transform layer.

| File | Raw table | Contents |
|------|-----------|----------|
| `patients.csv` | `raw_patients` | Demographics, cost totals |
| `encounters.csv` | `raw_encounters` | Clinical visits with costs |
| `conditions.csv` | `raw_conditions` | SNOMED-CT diagnoses |
| `medications.csv` | `raw_medications` | RxNorm prescriptions |
| `providers.csv` | `raw_providers` | Clinician details |

> **Why all-VARCHAR?** It guarantees no silent data loss from type coercion errors at ingest.
> We control exactly when and how types are applied — in the transform layer where the logic is testable.

> **TODO(future-source):** The `ingest/download.py` module has a documented swap point for replacing
> the Synthea CSV download with a Blue Button 2.0 FHIR R4 API consumer.

In [ ]:
from cms_platform.ingest.load import _TABLES

for table, (filename, cols) in _TABLES.items():
    print(f'{filename:30s} → {table}  ({len(cols)} columns)')

## Section 2 — Star Schema

Raw tables feed a **star schema** — a dimensional model with fact tables at the centre
and dimension tables around them.

```
dim_date ──────────────────────────────────────┐
dim_patient ──┬── fact_encounter ──── dim_provider
              ├── fact_condition
              └── fact_medication
dim_condition_code (SNOMED-CT dictionary)
```

**SNOMED-CT** codes identify conditions. SNOMED-CT is the international clinical terminology
standard used in clinical systems worldwide. It differs from ICD-10, which is used for US billing.
Real pipelines map SNOMED → ICD-10 at claim submission time.
> [Learn more about SNOMED-CT](https://www.snomed.org/snomed-ct/why-snomed-ct)

In [ ]:
import tempfile
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / 'tests'))

from test_ingest import _seed_csvs
from cms_platform.ingest.load import _ensure_raw_tables, load_synthea_data
from cms_platform.schema.transforms import build_star_schema
from cms_platform.common.config import Settings

# Build an in-memory DuckDB from fixture data
with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    settings = Settings(db_path=':memory:', raw_data_dir=str(tmp_path / 'raw'),
                        manifests_dir=str(tmp_path / 'manifests'))
    data_dir = _seed_csvs(tmp_path)
    conn = duckdb.connect()
    _ensure_raw_tables(conn)
    load_synthea_data(data_dir, conn)
    build_star_schema(conn, settings)

tables = ['dim_date', 'dim_patient', 'dim_provider', 'dim_condition_code',
          'fact_encounter', 'fact_condition', 'fact_medication']
for t in tables:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'{t:30s} {n:>8,} rows')

## Section 3 — Analytics

Five SQL queries answer the core healthcare analytics questions:

| Query | Key SQL technique |
|-------|------------------|
| 30-day readmission rate | `LAG` window function |
| Chronic-condition cohorts | Conditional aggregation (`CASE WHEN`) |
| Encounter cost distribution | `PERCENTILE_CONT`, `NTILE(4)` |
| Care-gap detection | `LEFT JOIN` anti-join |
| Utilisation year-over-year | `LAG` with `PARTITION BY` |

> Each SQL file carries a header documenting the business question, SQL technique,
> and how the query would be optimised at V2 (e.g., pre-aggregated materialised views).

In [ ]:
from cms_platform.analytics.queries import (
    readmission_30day, cohort_segmentation, cost_benchmarking,
    care_gap_detection, utilization_trends,
)

print('=== 30-day Readmissions ===')
print(readmission_30day(conn))

print('\n=== Cohort Segmentation ===')
print(cohort_segmentation(conn))

print('\n=== Cost Benchmarking ===')
print(cost_benchmarking(conn))

print('\n=== Care-Gap Detection ===')
print(care_gap_detection(conn))

print('\n=== Utilisation Trends ===')
print(utilization_trends(conn))

## Section 4 — Risk Stratification

A logistic regression pipeline predicts which patients are in the **top cost quartile**.
Features are derived from the star schema via a single SQL query.

> **Honest metrics note:** Synthetic data caps real predictive signal. These figures demonstrate
> the pipeline architecture, not clinical validity. A real deployment would require:
> - Validated clinical features (comorbidity indices, lab values)
> - Prospective outcome labelling
> - Calibration assessment and bias auditing

> **TODO(future-model):** The `scoring/risk_model.py` module has a documented swap point for
> replacing logistic regression with LightGBM once data volume justifies it.

In [ ]:
import random
import polars as pl
from cms_platform.scoring.risk_model import RISK_FEATURES, train_risk_model, predict_risk
from cms_platform.common.config import Settings

random.seed(42)
n = 50
rows = {col: [random.random() for _ in range(n)] for col in RISK_FEATURES}
df = pl.DataFrame(rows)
target = pl.Series('label', [i % 2 for i in range(n)])

settings = Settings()
model = train_risk_model(df, target, settings)
scores = predict_risk(model, df)

print(f'Training size: {model.training_size}')
print(f'Features:      {model.feature_cols}')
print(f'Score range:   {scores.min():.3f} – {scores.max():.3f}')
print()
print('Note: synthetic data caps real predictive signal.')
print('These figures demonstrate the pipeline, not clinical validity.')

## Section 5 — Care-Gap Explanation (Ollama)

For each patient with open care gaps, the platform generates a natural-language explanation
via [Ollama](https://ollama.com) (a local LLM server). When Ollama is unavailable,
the explainer falls back to a deterministic stub — so the demo runs fully offline.

> **What is Ollama?** A tool for running open-source LLMs (like Llama 3.2) locally on your machine.
> It exposes an OpenAI-compatible API, so the same client code works with any provider.
> [ollama.com](https://ollama.com)

In [ ]:
from cms_platform.scoring.explainer import explain_care_gaps
from cms_platform.common.config import Settings

settings = Settings()
patient_id = 'aaaaaaaa-0001-0001-0001-000000000001'
gaps = ['Diabetes mellitus type 2 — no encounter in past 12 months']

explanation = explain_care_gaps(patient_id, gaps, settings)
print(f'Patient:    {patient_id}')
print(f'Gaps:       {explanation.gaps}')
print(f'Model used: {explanation.model_used}')
print(f'Summary:    {explanation.summary}')

## Section 6 — V0 → V2 Architecture

Each layer has a documented swap point for the V2 migration:

| Layer | V0 (now) | V2 (future) | Swap point |
|-------|----------|-------------|------------|
| Storage | DuckDB (embedded) | Postgres + columnar | `common/db.py::get_connection()` |
| Ingest | Synthea CSV download | Blue Button 2.0 FHIR API | `ingest/download.py` |
| Streaming | None | Kafka (one topic per resource type) | `ingest/load.py` |
| API scale | Single FastAPI container | LB + replicas | `api/main.py` |
| Risk model | Logistic regression | LightGBM | `scoring/risk_model.py` |

**Real-data path:** To ingest real Medicare data, register a CMS Blue Button 2.0 application
at [bluebutton.cms.gov](https://bluebutton.cms.gov). The FHIR R4 API returns `Patient`,
`ExplanationOfBenefit`, and `Coverage` resources — these map directly to the raw table schema.

> The principle: defer complexity until the pain is real and measurable.
> V0 runs on a laptop. V2 runs on a cluster. The migration path between them is explicit and non-destructive.

## Repository map

```
src/cms_platform/
├── common/          # config, db, audit, mask, logging
├── ingest/          # download.py (Synthea), load.py (raw tables)
├── schema/          # transforms.py (raw → star schema)
├── analytics/       # queries.py (5 analytical queries)
├── scoring/         # risk_model.py, explainer.py
└── api/             # FastAPI app — routes, models, deps

sql/
├── schema/ddl.sql   # Star schema DDL
└── analytics/       # 5 analytical SQL files

tests/               # Unit tests — one file per module
notebooks/           # This narrative notebook
```

See [`ARCHITECTURE.md`](../ARCHITECTURE.md) for the full V2 distributed-systems design,
including Kafka streaming, multi-tenancy, schema evolution, and cost model.